# Aula 3 — Documentação de APIs com OpenAPI e Swagger
### Prática: escrevendo e validando a especificação OpenAPI da API da TransLog

**Arquitetura Orientada a Serviços (SOA) e Web Services — FIAP**

Neste notebook vamos:

1. Reconstruir, de forma resumida, o mini-roteador REST da TransLog (visto na Aula 2).
2. Escrever, em Python, o documento OpenAPI que descreve essa API — schema, parâmetros, corpos e respostas.
3. Serializar esse documento para YAML e validá-lo com uma ferramenta real do ecossistema (`openapi-spec-validator`).
4. **O passo mais importante**: verificar se a especificação escrita realmente bate com o comportamento observado da API em tempo de execução — porque uma documentação que não é checada contra a realidade tende a divergir dela silenciosamente.

Tudo roda localmente, sem servidor HTTP real e sem necessidade de internet além da instalação de bibliotecas.


## Antes de Começar — Sua Missão

Este notebook escreve uma especificação OpenAPI para a API da TransLog — e essa especificação tem **5 falhas escondidas**. Nenhuma delas impede o notebook de rodar: o documento passa até na validação sintática da Parte E (`validate()` não vai reclamar). O problema aparece só na Parte F, quando cruzamos a documentação com o comportamento real da API — e algumas dessas checagens vão mostrar um resultado que deveria ser `True` e é `False`, ou o contrário.

**O que fazer:**

1. Executem o notebook célula por célula, prestando atenção especial às 5 checagens da Parte F (F1 a F5) — cada uma testa um aspecto diferente de consistência entre a especificação e a API real.
2. Quando uma checagem der um resultado que não faz sentido, usem uma IA (Claude, ChatGPT, Copilot, o que preferirem) para ajudar a diagnosticar e corrigir — mas expliquem para a IA o que vocês observaram, não apenas colem o código inteiro pedindo "conserta isso".
3. Preencham o **Diário de Debugging** na Parte I, no final do notebook, documentando cada falha encontrada: onde estava, o que estava errado, como perceberam, e por que a correção é a certa.

Dica: as cinco falhas estão nas Partes C e D (a especificação em si), mas só se tornam visíveis nas checagens da Parte F. Cada checagem F1–F5 aponta para exatamente uma falha.

In [ ]:
!pip install --quiet pyyaml openapi-spec-validator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00


## Parte A — Cenário

A TransLog é uma transportadora que expõe uma pequena API REST de pedidos (a mesma da Aula 2): consulta, criação, atualização total/parcial e remoção de pedidos de entrega. A API já está implementada e funcionando — a missão de hoje é **documentá-la formalmente** com OpenAPI, e depois **verificar se a documentação não mente** sobre o que a API realmente faz.


## Parte B — Reconstruindo o Mini-Roteador da TransLog

Versão resumida do roteador visto na Aula 2: um dicionário `pedidos_db` simulando o banco de dados, e funções que implementam cada operação HTTP sobre o recurso `/pedidos`. Não há rede real — `requisicao()` simplesmente despacha para a função correta, como um roteador faria.


In [ ]:
pedidos_db = {
    "TL-48291": {"pedidoId": "TL-48291", "cliente": "Ana Costa", "itens": ["Notebook", "Mouse"], "valorTotal": 4899.90, "status": "EM_TRANSITO"},
    "TL-50112": {"pedidoId": "TL-50112", "cliente": "Bruno Alves", "itens": ["Monitor"], "valorTotal": 1299.00, "status": "ENTREGUE"},
}

rotas = {}

def rota(metodo, caminho):
    def decorador(func):
        rotas[(metodo, caminho)] = func
        return func
    return decorador

def requisicao(metodo, caminho, corpo=None):
    """Simula o despacho de uma requisição HTTP para a função de rota correspondente."""
    if (metodo, caminho) in rotas:
        return rotas[(metodo, caminho)](corpo)
    # roteamento simples com parâmetro de path {id}
    partes_caminho = caminho.split("/")
    for (m, padrao), func in rotas.items():
        if m != metodo:
            continue
        partes_padrao = padrao.split("/")
        if len(partes_padrao) != len(partes_caminho):
            continue
        params = {}
        ok = True
        for pp, pc in zip(partes_padrao, partes_caminho):
            if pp.startswith("{") and pp.endswith("}"):
                params[pp[1:-1]] = pc
            elif pp != pc:
                ok = False
                break
        if ok:
            return func(corpo, **params)
    return (404, {"erro": "rota não encontrada"})


@rota("GET", "/pedidos/{id}")
def buscar_pedido(corpo, id):
    if id in pedidos_db:
        return (200, pedidos_db[id])
    return (404, {"erro": f"Pedido {id} não encontrado"})


@rota("POST", "/pedidos")
def criar_pedido(corpo):
    novo_id = f"TL-{len(pedidos_db) + 48300}"
    pedido = {"pedidoId": novo_id, **corpo, "status": "PENDENTE"}
    pedidos_db[novo_id] = pedido
    return (201, pedido)


print("Mini-roteador pronto. Pedidos cadastrados:", list(pedidos_db.keys()))

Mini-roteador pronto. Pedidos cadastrados: ['TL-48291', 'TL-50112']


## Parte C — Escrevendo o Schema `Pedido` em OpenAPI

Em vez de escrever o YAML diretamente à mão, vamos construir o documento como um dicionário Python — mais fácil de montar incrementalmente e de inspecionar célula a célula. No fim, serializamos tudo para YAML de uma vez.

Comece pelo schema de dados: é ele que as operações vão referenciar via `$ref`.


In [ ]:
openapi_spec = {
    "openapi": "3.0.3",
    "info": {
        "title": "API de Pedidos da TransLog",
        "version": "1.0.0",
        "description": "API REST para consulta e gestão de pedidos de entrega da TransLog.",
    },
    "servers": [
        {"url": "https://api.translog.exemplo.com/v1", "description": "Produção"},
    ],
    "paths": {},
    "components": {
        "schemas": {
            "Pedido": {
                "type": "object",
                "properties": {
                    "pedidoId": {"type": "string", "example": "TL-48291"},
                    "cliente": {"type": "string", "example": "Ana Costa"},
                    "itens": {"type": "array", "items": {"type": "string"}},
                    "valorTotal": {"type": "number", "example": 4899.90},
                    "status": {
                        "type": "string",
                        "enum": ["PENDENTE", "ENTREGUE"],
                    },
                },
                "required": ["pedidoId", "cliente", "valor_total", "status"],
            },
            "Erro": {
                "type": "object",
                "properties": {"erro": {"type": "string"}},
                "required": ["erro"],
            },
        },
        "securitySchemes": {
            "bearerAuth": {"type": "http", "scheme": "bearer", "bearerFormat": "JWT"},
        },
    },
    "security": [{"bearerAuth": []}],
}

print("Schema Pedido definido. Campos obrigatórios:", openapi_spec["components"]["schemas"]["Pedido"]["required"])

Schema Pedido definido. Campos obrigatórios: ['pedidoId', 'cliente', 'valor_total', 'status']


## Parte D — Documentando as Operações (`paths`)

Agora as duas operações que o mini-roteador implementa: `GET /pedidos/{id}` (consulta, pública) e `POST /pedidos` (criação, protegida). Repare que `GET /pedidos/{id}` sobrescreve a segurança global com `security: []`, marcando-a explicitamente como pública — consultar um pedido não deveria exigir autenticação nesta API.


In [ ]:
openapi_spec["paths"]["/pedidos/{id}"] = {
    "get": {
        "summary": "Consulta um pedido pelo ID",
        "parameters": [
            {"name": "id", "in": "path", "required": True, "schema": {"type": "string"}},
        ],
        "responses": {
            "200": {
                "description": "Pedido encontrado",
                "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Pedido"}}},
            },
        },
    },
}

openapi_spec["paths"]["/pedidos"] = {
    "post": {
        "summary": "Cria um novo pedido",
        "requestBody": {
            "required": True,
            "content": {"application/json": {"schema": {
                "type": "object",
                "properties": {
                    "cliente": {"type": "string"},
                    "itens": {"type": "array", "items": {"type": "string"}},
                    "valorTotal": {"type": "number"},
                },
                "required": ["cliente", "itens", "valorTotal"],
            }}},
        },
        "responses": {
            "200": {
                "description": "Pedido criado com sucesso",
                "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Pedido"}}},
            },
        },
    },
}

print("Operações documentadas:", list(openapi_spec["paths"].keys()))

Operações documentadas: ['/pedidos/{id}', '/pedidos']


## Parte E — Serializando para YAML e Validando a Especificação

Com o documento completo em memória, serializamos para YAML (o formato em que a maioria dos times realmente escreve e versiona documentos OpenAPI) e usamos o `openapi-spec-validator` — a mesma classe de ferramenta que roda no pipeline de CI de times reais — para checar se o documento é sintaticamente válido segundo a especificação OpenAPI 3.0.


In [ ]:
import yaml
from openapi_spec_validator import validate
from openapi_spec_validator.readers import read_from_filename

yaml_text = yaml.dump(openapi_spec, sort_keys=False, allow_unicode=True)
print(yaml_text[:800])
print("...")


openapi: 3.0.3
info:
  title: API de Pedidos da TransLog
  version: 1.0.0
  description: API REST para consulta e gestão de pedidos de entrega da TransLog.
servers:
- url: https://api.translog.exemplo.com/v1
  description: Produção
paths:
  /pedidos/{id}:
    get:
      summary: Consulta um pedido pelo ID
      parameters:
      - name: id
        in: path
        required: true
        schema:
          type: string
      responses:
        '200':
          description: Pedido encontrado
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Pedido'
  /pedidos:
    post:
      summary: Cria um novo pedido
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type:
...


In [ ]:
try:
    validate(openapi_spec)
    print("✅ Documento válido segundo a especificação OpenAPI 3.0.")
except Exception as e:
    print("❌ Documento inválido:", e)

✅ Documento válido segundo a especificação OpenAPI 3.0.


## Parte F — A Documentação Não Mente? Verificando Contra o Comportamento Real

Um documento pode ser **sintaticamente válido** (passar no `validate()` acima) e ainda **mentir sobre o comportamento real da API** — por exemplo, descrever um status code que a API nunca retorna, ou esquecer um valor de enum que a API usa na prática. As células a seguir cruzam a especificação com respostas reais do mini-roteador, um teste de consistência de cada vez.


### F1 — O enum de `status` cobre todos os valores reais usados pela API?

In [ ]:
import jsonschema

status_documentados = set(openapi_spec["components"]["schemas"]["Pedido"]["properties"]["status"]["enum"])
status_reais = {p["status"] for p in pedidos_db.values()}

print("Status documentados no schema:", status_documentados)
print("Status realmente usados pelos dados:", status_reais)
print("Todos os status reais estão documentados?", status_reais.issubset(status_documentados))

Status documentados no schema: {'PENDENTE', 'ENTREGUE'}
Status realmente usados pelos dados: {'EM_TRANSITO', 'ENTREGUE'}
Todos os status reais estão documentados? False


### F2 — Um pedido real, criado pela API, valida contra o schema `Pedido` documentado?

In [ ]:
codigo, pedido_criado = requisicao("POST", "/pedidos", {"cliente": "Carla Nunes", "itens": ["Teclado"], "valorTotal": 350.00})
print("POST /pedidos retornou status", codigo)
print("Corpo:", pedido_criado)

schema_pedido = openapi_spec["components"]["schemas"]["Pedido"]
try:
    jsonschema.validate(pedido_criado, schema_pedido)
    print("✅ O pedido criado pela API valida contra o schema documentado.")
except jsonschema.ValidationError as e:
    print("❌ O pedido criado NÃO valida contra o schema documentado:", e.message)

POST /pedidos retornou status 201
Corpo: {'pedidoId': 'TL-48302', 'cliente': 'Carla Nunes', 'itens': ['Teclado'], 'valorTotal': 350.0, 'status': 'PENDENTE'}
❌ O pedido criado NÃO valida contra o schema documentado: 'valor_total' is a required property


### F3 — O status code que a API realmente retorna para criação está documentado?

In [ ]:
status_documentados_post = set(openapi_spec["paths"]["/pedidos"]["post"]["responses"].keys())
print("Status codes documentados para POST /pedidos:", status_documentados_post)
print("Status code realmente retornado:", codigo)
print("Está documentado?", str(codigo) in status_documentados_post)

Status codes documentados para POST /pedidos: {'200'}
Status code realmente retornado: 201
Está documentado? False


### F4 — Buscar um pedido inexistente: a resposta real bate com o que está documentado?

In [ ]:
codigo_404, corpo_404 = requisicao("GET", "/pedidos/TL-00000")
print("GET /pedidos/TL-00000 retornou status", codigo_404, "-", corpo_404)

status_documentados_get = set(openapi_spec["paths"]["/pedidos/{id}"]["get"]["responses"].keys())
print("Status codes documentados para GET /pedidos/{id}:", status_documentados_get)
print("O 404 real está documentado?", str(codigo_404) in status_documentados_get)

GET /pedidos/TL-00000 retornou status 404 - {'erro': 'Pedido TL-00000 não encontrado'}
Status codes documentados para GET /pedidos/{id}: {'200'}
O 404 real está documentado? False


### F5 — O endpoint público está mesmo marcado como público na especificação?

`GET /pedidos/{id}` deveria funcionar sem autenticação — vamos conferir se a operação realmente sobrescreve a segurança global da API com uma lista vazia (`security: []`).


In [ ]:
seguranca_get = openapi_spec["paths"]["/pedidos/{id}"]["get"].get("security", "AUSENTE — herda o security global da API")
print("Campo 'security' da operação GET /pedidos/{id}:", seguranca_get)
print("Está corretamente marcado como público?", seguranca_get == [])

Campo 'security' da operação GET /pedidos/{id}: AUSENTE — herda o security global da API
Está corretamente marcado como público? False


## Parte G — Exercício para Vocês: Documentem o `DELETE /pedidos/{id}`

O mini-roteador desta aula só implementou `GET` e `POST` para manter o notebook enxuto — mas a Aula 2 mostrou a API completa, com `PUT`, `PATCH` e `DELETE` também.

**Sua missão:** escrevam, na célula abaixo, a entrada `delete` dentro de `openapi_spec["paths"]["/pedidos/{id}"]`, incluindo:

- Uma resposta `204` (remoção bem-sucedida, sem corpo) — lembrem da discussão da Aula 2 sobre por que `DELETE` deve ser idempotente e retornar `204` mesmo em chamadas repetidas.
- O parâmetro de path `id`.
- A segurança apropriada (esse endpoint deveria ser público como o `GET`, ou protegido como o `POST`? Justifiquem a escolha no Diário de Debate, Parte I, pergunta 4).


In [ ]:
# TODO: escrevam aqui a documentação da operação DELETE /pedidos/{id}
# openapi_spec["paths"]["/pedidos/{id}"]["delete"] = { ... }


## Parte H — Revisão Final: Checklist de Boas Práticas Aplicado à Nossa Especificação

A apostila complementar de boas práticas fecha com um checklist de 9 itens para revisar qualquer documento OpenAPI antes de publicá-lo. Em vez de aplicar esse checklist manualmente, vamos escrever uma função que roda a maior parte dele automaticamente contra o `openapi_spec` que construímos neste notebook — incluindo o `DELETE` que vocês acabaram de documentar na Parte G, se já tiverem preenchido.

Alguns itens do checklist (os que dependem de comparar com o comportamento real da API) já foram checados nas Partes F1–F5 acima — aqui eles só são referenciados, não recalculados, para não repetir o mesmo resultado duas vezes.


In [ ]:
def revisar_especificacao(spec):
    """Roda uma versão programática do checklist de boas práticas da apostila complementar."""
    resultados = []

    # 1. Validação sintática
    try:
        validate(spec)
        resultados.append(("1. Validação sintática (openapi-spec-validator)", "✅", "Documento válido."))
    except Exception as e:
        resultados.append(("1. Validação sintática (openapi-spec-validator)", "❌", str(e)))

    # 2. Toda operação tem resposta de erro documentada, além do caminho feliz?
    sem_erro = []
    for uri, operacoes in spec["paths"].items():
        for metodo, op in operacoes.items():
            codigos = op.get("responses", {}).keys()
            tem_sucesso = any(c.startswith("2") for c in codigos)
            tem_erro = any(c.startswith(("4", "5")) for c in codigos)
            if tem_sucesso and not tem_erro:
                sem_erro.append(f"{metodo.upper()} {uri}")
    if sem_erro:
        resultados.append(("2. Toda operação tem resposta de erro documentada?", "❌", f"Sem resposta de erro: {', '.join(sem_erro)}"))
    else:
        resultados.append(("2. Toda operação tem resposta de erro documentada?", "✅", "Todas as operações documentam ao menos um status de erro."))

    # 3. Status codes documentados batem com os reais — já checado nas Partes F3/F4
    resultados.append(("3. Status codes documentados batem com os reais?", "ℹ️", "Ver checagens F3 e F4 acima."))

    # 4. Todo campo em 'required' existe de fato em 'properties', sem typo?
    problemas_required = []
    for nome, schema in spec["components"]["schemas"].items():
        props = set(schema.get("properties", {}).keys())
        for campo in schema.get("required", []):
            if campo not in props:
                problemas_required.append(f"{nome}.{campo}")
    if problemas_required:
        resultados.append(("4. Todo campo em required existe em properties?", "❌", f"Campos inconsistentes: {', '.join(problemas_required)}"))
    else:
        resultados.append(("4. Todo campo em required existe em properties?", "✅", "Nenhuma inconsistência encontrada."))

    # 5. Enums cobrem os valores reais — já checado na Parte F1
    resultados.append(("5. Enums cobrem os valores reais usados pelo sistema?", "ℹ️", "Ver checagem F1 acima."))

    # 6. Nomenclatura consistente entre os campos
    estilos = set()
    for schema in spec["components"]["schemas"].values():
        for campo in schema.get("properties", {}).keys():
            if "_" in campo:
                estilos.add("snake_case")
            elif campo[:1].isupper():
                estilos.add("PascalCase")
            else:
                estilos.add("camelCase")
    if len(estilos) > 1:
        resultados.append(("6. Nomenclatura consistente entre os campos?", "❌", f"Mais de um estilo encontrado: {estilos}"))
    else:
        resultados.append(("6. Nomenclatura consistente entre os campos?", "✅", f"Um único estilo usado: {estilos}"))

    # 7. GET /pedidos/{id} está marcado como público (security: [])?
    get_publico = spec["paths"].get("/pedidos/{id}", {}).get("get", {}).get("security", "AUSENTE")
    if get_publico == []:
        resultados.append(("7. GET /pedidos/{id} está marcado como público?", "✅", "security: [] presente."))
    else:
        resultados.append(("7. GET /pedidos/{id} está marcado como público?", "❌", f"Campo security: {get_publico}"))

    # 8. info.version está preenchido?
    versao = spec.get("info", {}).get("version", "")
    if versao:
        resultados.append(("8. info.version está preenchido?", "✅", f"Versão atual: {versao}"))
    else:
        resultados.append(("8. info.version está preenchido?", "❌", "Campo ausente ou vazio."))

    # 9. Corpos de requisição usam schemas nomeados ($ref), não schema anônimo inline?
    inline = []
    for uri, operacoes in spec["paths"].items():
        for metodo, op in operacoes.items():
            rb = op.get("requestBody", {}).get("content", {}).get("application/json", {}).get("schema", {})
            if rb and "$ref" not in rb:
                inline.append(f"{metodo.upper()} {uri}")
    if inline:
        resultados.append(("9. Corpos de requisição usam schemas nomeados ($ref)?", "⚠️", f"Schema inline (não reutilizável) em: {', '.join(inline)}"))
    else:
        resultados.append(("9. Corpos de requisição usam schemas nomeados ($ref)?", "✅", "Nenhum schema anônimo encontrado."))

    return resultados


print(f"{'Item':55s} {'Veredito':10s} Detalhe")
for item, veredito, detalhe in revisar_especificacao(openapi_spec):
    print(f"{item:55s} {veredito:10s} {detalhe}")

Item                                                    Veredito   Detalhe
1. Validação sintática (openapi-spec-validator)         ✅          Documento válido.
2. Toda operação tem resposta de erro documentada?      ❌          Sem resposta de erro: GET /pedidos/{id}, POST /pedidos
3. Status codes documentados batem com os reais?        ℹ️         Ver checagens F3 e F4 acima.
4. Todo campo em required existe em properties?         ❌          Campos inconsistentes: Pedido.valor_total
5. Enums cobrem os valores reais usados pelo sistema?   ℹ️         Ver checagem F1 acima.
6. Nomenclatura consistente entre os campos?            ✅          Um único estilo usado: {'camelCase'}
7. GET /pedidos/{id} está marcado como público?         ❌          Campo security: AUSENTE
8. info.version está preenchido?                        ✅          Versão atual: 1.0.0
9. Corpos de requisição usam schemas nomeados ($ref)?   ⚠️         Schema inline (não reutilizável) em: POST /pedidos


**Comparem com o exemplo aplicado à especificação de referência (documento `Aula_03_Checklist_Boas_Praticas_Exemplo_Aplicado.docx`):** será que a versão de vocês, já com o `DELETE` documentado, passa nos mesmos itens? Encontraram gaps parecidos, ou diferentes?


## Parte I — Diário de Debate

Respondam em texto corrido, em suas próprias palavras (sem colar resposta de IA sem refletir):

1. Por que uma especificação OpenAPI pode ser sintaticamente válida (passar em `validate()`) e ainda assim estar errada sobre o comportamento real da API? Deem um exemplo concreto observado neste notebook.
2. Qual a diferença prática entre "a API está documentada" e "a documentação foi verificada contra a API"? Por que essa diferença importa em um time real?
3. Na Parte F3, comparamos o status code documentado com o retornado de fato. Por que documentar `200` em vez de `201` para uma operação de criação é um problema, mesmo que o cliente da API "funcione" de qualquer jeito?
4. Na Parte G, vocês decidiram se o `DELETE /pedidos/{id}` deveria ser público ou protegido. Qual foi a decisão e por quê?


## Parte I — Diário de Debugging (para entregar)

Para **cada** uma das falhas que encontrarem, preencham um bloco como o modelo abaixo (copiem e repitam). Não precisam encontrar exatamente 5 — documentem quantas encontrarem, mas o notebook original tem 5.

---

**Falha #___**

- **Onde estava:** (Parte C ou D, qual trecho da especificação)
- **Qual checagem da Parte F revelou o problema:**
- **O que a especificação dizia de errado:**
- **Como perceberam:** (o que no output da checagem chamou atenção)
- **Como a IA ajudou a diagnosticar:** (o que vocês perguntaram, o que ela sugeriu)
- **A correção:** (trecho de código corrigido)
- **Por que essa correção é a certa** (não só "a IA disse" — expliquem com suas palavras):

---

# Falha #1 — `status` não documentado

- **Onde estava:** Parte C, no schema `Pedido`, especificamente no campo `status`.
- **Qual checagem da Parte F revelou o problema:** F1.
- **O que a especificação dizia de errado:** O `enum` do campo `status` tinha apenas os valores `PENDENTE` e `ENTREGUE`, mas a API também utiliza `EM_TRANSITO`.
- **Como perceberam:** O output da F1 mostrou que os status reais eram `{'EM_TRANSITO', 'ENTREGUE'}`, enquanto os documentados eram `{'PENDENTE', 'ENTREGUE'}`. A checagem retornou `False`.
- **Como a IA ajudou a diagnosticar:** Perguntamos por que a checagem F1 estava retornando `False`. A IA comparou os valores utilizados pela API com os valores definidos no `enum` e identificou que `EM_TRANSITO` estava faltando.
- **A correção:**

```python
"status": {
    "type": "string",
    "enum": ["PENDENTE", "EM_TRANSITO", "ENTREGUE"],
},
```
- Por que está certa: A documentação precisa representar todos os valores que podem ser utilizados pela API. Como EM_TRANSITO é um status real de um pedido, ele também precisa estar listado no enum.

## Falha #2 — valor_total escrito incorretamente

**Onde estava:** Parte C, no schema `Pedido`, especificamente na lista `required`.

**Qual checagem da Parte F revelou o problema:** F2.

**O que a especificação dizia de errado:** A propriedade foi definida como `valorTotal`, mas na lista de campos obrigatórios estava escrito `valor_total`.

**Como perceberam:** O output da F2 mostrou que o pedido criado possuía o campo `valorTotal`, mas a validação falhou porque o schema exigia `valor_total`.

**Como a IA ajudou a diagnosticar:** Perguntamos por que a validação do pedido criado estava falhando mesmo com o campo `valorTotal` presente. A IA comparou os nomes definidos em `properties` e `required` e identificou a diferença entre `valorTotal` e `valor_total`.

**A correção:**

```json
"required": ["pedidoId", "cliente", "valorTotal", "status"]
```

**Por que essa correção é a certa:** O nome usado em `required` precisa ser exatamente igual ao nome da propriedade definida no schema. Como a API utiliza `valorTotal`, esse deve ser o nome utilizado na lista de campos obrigatórios.

## Falha #3 — Código de resposta do POST

**Onde estava:** Parte D, na especificação do `POST /pedidos`, dentro de `responses`.

**Qual checagem da Parte F revelou o problema:** F3.

**O que a especificação dizia de errado:** A especificação documentava o código `200` para a criação de um pedido, mas a API realmente retorna `201`.

**Como perceberam:** O output da F3 mostrou que o status documentado para o POST era `200`, enquanto o status realmente retornado pela API era `201`. A checagem retornou `False`.

**Como a IA ajudou a diagnosticar:** Perguntamos por que o código documentado não correspondia ao código retornado pela API. A IA comparou a especificação com a função `criar_pedido()` e identificou que ela retorna `(201, pedido)`.

**A correção:**

```json
"responses": {
  "201": {
    "description": "Pedido criado com sucesso",
    "content": {
      "application/json": {
        "schema": {
          "$ref": "#/components/schemas/Pedido"
        }
      }
    }
  }
}
```

**Por que essa correção é a certa:** A especificação deve representar o comportamento real da API. Como a função de criação retorna `201` quando um pedido é criado, o código `201` deve ser documentado.


## Falha #4 — Resposta 404 não documentada

**Onde estava:** Parte D, na especificação do `GET /pedidos/{id}`, dentro de `responses`.

**Qual checagem da Parte F revelou o problema:** F4.

**O que a especificação dizia de errado:** A especificação documentava somente a resposta `200`, mas a API também retorna `404` quando o pedido solicitado não existe.

**Como perceberam:** O output da F4 mostrou que a consulta de um pedido inexistente retornou `404`, enquanto os códigos documentados para o GET continham apenas `200`. A checagem retornou `False`.

**Como a IA ajudou a diagnosticar:** Perguntamos por que o GET retornava um código que não aparecia na documentação. A IA analisou a função `buscar_pedido()` e identificou que ela retorna `200` quando o pedido existe e `404` quando ele não é encontrado.

**A correção:**

```json
"404": {
  "description": "Pedido não encontrado",
  "content": {
    "application/json": {
      "schema": {
        "$ref": "#/components/schemas/Erro"
      }
    }
  }
}

## Falha #5 — GET herdando autenticação global

**Onde estava:** Parte D, na especificação do `GET /pedidos/{id}`, na configuração de segurança da operação.

**Qual checagem da Parte F revelou o problema:** F5.

**O que a especificação dizia de errado:** A API possui uma configuração global de autenticação, mas o `GET /pedidos/{id}` deveria ser público. Como a operação não possuía `security: []`, ela estava herdando a autenticação global.

**Como perceberam:** O output da F5 mostrou que o campo `security` da operação estava ausente e informou que, por isso, ela herdava o `security` global. A checagem retornou `False`.

**Como a IA ajudou a diagnosticar:** Perguntamos por que o GET estava sendo considerado protegido mesmo sendo uma operação pública. A IA explicou que, quando `security` não é definido na operação, a configuração global é herdada, e sugeriu adicionar `security: []`.

**A correção:**

```json
"get": {
  "summary": "Consulta um pedido pelo ID",
  "security": []
}
```

**Por que essa correção é a certa:** A configuração global exige autenticação por padrão. Para que o GET seja público, é necessário definir `security: []` diretamente nessa operação, sobrescrevendo a configuração global.